# 16. Compare baselines on the same held-out source videos

Does the new score add value beyond local movement, extra capacity and inexpensive models? We will reconstruct the declared source-level estimator, inspect paired uncertainty and separate normal-only detection from supervised condition classification.

## Run this notebook independently

From the repository root, install the shared environment and start Jupyter:

```bash
uv sync --project experiments/aaai
AAAI_ITERATION02_RUN_DIR=/absolute/path/to/experiments/aaai/artifacts/iteration02/PROTOCOL_HASH \
  uv run --project experiments/aaai jupyter lab experiments/aaai
```

Replace `PROTOCOL_HASH` with the exact completed run you intend to inspect. The setup displays its producer command and verifies its recorded identity before showing results. It never chooses the newest run automatically. Alternatively, put `AAAI_ITERATION02_RUN_DIR` in the repository root `.env`. Do not create another environment file. Python-dotenv calls the real functions `load_dotenv` and `find_dotenv`; the aliases below implement the requested `load_env(find_env())` spelling without printing credentials.

Python 3.12 and the notebook, plotting, data and model dependencies are declared in the existing `pyproject.toml`. Select its kernel. This notebook does not depend on cells from another notebook. A full experiment fits models through the recorded producer command; this tutorial loads those real fitted outputs, checks them and teaches how to interpret them. Missing required outputs stop execution instead of triggering synthetic examples.

A small development run has its own protocol identity. To inspect it deliberately, also set `AAAI_ITERATION02_ALLOW_DEVELOPMENT=1`. Its outputs remain labeled incomplete. Do not use that option to bypass a failed evidence verifier.

The experiment is exploratory because the available sources informed earlier work. A dataset folder label is not a diagnosis independently established here. The focal endpoint is normal versus Parkinsons, stroke, cerebral palsy and myopathic labels. The mandatory broad endpoint uses all non-normal folders. The word “normal” refers to the dataset annotation, not a clinical clearance.

In [ ]:
from pathlib import Path
import os, sys, json
from dotenv import load_dotenv as load_env, find_dotenv as find_env

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ambient").is_dir() and (p / "data/gavd").is_dir())
os.chdir(ROOT)
load_env(find_env())
BASE = ROOT / "experiments/aaai"
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.iteration02_notebook_support import (
    EvidenceRun, package_versions, show_diagram, show_video, show_status,
    optional_evidence, joint_table, source_role_table, plot_counts,
    prediction_example, example_losses, plot_prediction, plot_adequacy,
    plot_timing, plot_source_scores, within_fold_metrics, plot_metric_table,
    fold_json_records, fold_tables, cohort_table, mask_bank_summary,
    measured_window, plot_paired_differences, timing_video_example, external_replication, plot_condition_confusions,
)

run = EvidenceRun.from_env()
display(run.describe().T.rename(columns={0: "value"}))
display(package_versions())
if run.mode not in {"full", "full_study"}:
    display(Markdown("**DEVELOPMENT RUN. This is incomplete evidence and cannot supply final manuscript results.**"))

## 1. Use the same sources and frozen questions

The focal primary endpoint is normal versus the four requested condition families. The broad secondary endpoint uses all current non-normal folders, including activity and style labels. Both use the same frozen source scores. This focal redesign followed historical all-folder results and remains exploratory.

Compare methods only on their declared matched population. A method should not gain a favorable denominator by silently dropping videos it cannot score. Report unsupported required comparisons in the status ledger. A published method name is justified only by a verified reproduction or a clearly described adaptation.

In [ ]:
registry = run.table("source_registry.csv")
display(source_role_table(registry).groupby(["outer_fold", "role"]).source_id.nunique().unstack(fill_value=0))
show_status(run, "baseline_status.json")
show_video(run, label="normal")

## 2. Reconstruct the within-fold AUROC by hand

Within one outer fold, AUROC asks whether an abnormal-labeled source receives a higher score than a normal-labeled source. Average the fold AUROCs with fixed weights n_normal multiplied by n_abnormal. These weights count the comparable pairs within each fold.

Do not pool differently calibrated fold scores to obtain the primary estimate. Average precision is reported per fold together with prevalence because its scale depends on the abnormal proportion. Four folds are not four independent clinical studies.

In [ ]:
scores = run.table("source_scores.csv", columns=("outer_fold", "source_id", "policy", "bank", "binary_label", "focal_eligible", "M", "R", "S"))
primary_bank = "U" if "U" in scores.bank.unique() else "E_U"
reference = scores.loc[scores.policy.eq("U") & scores.bank.eq(primary_bank)]
recomputed = []
for endpoint in ("focal", "broad"):
    for score_name in ("M", "S"):
        fold_table, value = within_fold_metrics(reference, score_name, endpoint=endpoint)
        display(fold_table.assign(endpoint=endpoint, score=score_name))
        recomputed.append({"endpoint": endpoint, "score": score_name, "pair_weighted_auroc": value})
recomputed = pd.DataFrame(recomputed)
display(recomputed)
saved_metrics = run.table("metrics.csv")
for item in recomputed.itertuples():
    match = saved_metrics.loc[saved_metrics.policy.eq("U") & saved_metrics.bank.eq(primary_bank)
                              & saved_metrics.endpoint.eq(item.endpoint) & saved_metrics.method.eq(item.score)
                              & saved_metrics.metric.eq("auroc")]
    assert len(match) == 1 and np.isclose(match.value.iloc[0], item.pair_weighted_auroc, atol=1e-12)
print("Every reconstructed M/S AUROC matches the saved estimator.")

## 3. Read paired uncertainty, not just an attractive point estimate

The paired bootstrap resamples whole source records within each fold and binary-label stratum. Each sampled source carries every method score together, and the original fold weights stay fixed. Confirmed duplicate-recording groups must be treated as a shared resampling cluster.

The interval is conditional on the fitted models, source split and selected protocol. It does not include training-seed uncertainty, unknown repeated-person dependence or the earlier adaptive redesign. An interval crossing zero does not establish a detection advantage. A narrow software-reproducibility check is not a replacement for independent clinical evidence.

In [ ]:
metrics = run.table("metrics.csv", columns=("metric", "method", "endpoint", "value"))
display(metrics)
display(pd.json_normalize(run.json("paired_intervals.json")))
display(plot_paired_differences(run)); plt.close("all")
raw_metrics = run.json("rawmetrics.json")
display(pd.json_normalize(raw_metrics).T if isinstance(raw_metrics, dict) else raw_metrics)

## 4. Report all training masks within each evaluation bank

A versus U on E_U is the primary masking-policy contrast. A versus J helps distinguish anatomical co-masking from simply hiding one joint's entire history. J versus U is a secondary explanation. Keep all arms, including failed adequacy gates.

If an alternative improves only E_leg, describe it as specialization to that missing-context task. Do not replace the original result with whichever bank looks better. A diagnostic with too few supported normal references must remain unavailable.

In [ ]:
masking_rows = []
for (bank, policy), group in scores.groupby(["bank", "policy"], sort=True):
    for endpoint in ("focal", "broad"):
        try:
            folds, value = within_fold_metrics(group, "S", endpoint=endpoint)
        except ValueError as error:
            masking_rows.append({"bank": bank, "policy": policy, "endpoint": endpoint,
                                 "status": "unavailable", "reason": str(error)})
        else:
            masking_rows.append({"bank": bank, "policy": policy, "endpoint": endpoint,
                                 "status": "measured", "sources": len(group), "auroc": value})
display(pd.DataFrame(masking_rows))
display(plot_metric_table(run)); plt.close("all")

## 5. Make simple alternatives difficult to dismiss

The own-leg capacity control asks whether flexibility explains the change. Shared ridge and a small temporal model ask whether a complex representation is needed. Local movement and valid phase descriptors ask whether simpler measurements already explain the score. A random encoder with the same coordinate readout tests whether representation learning adds value.

A failed or unimplemented JEPA branch cannot be described as a successful learned world model. The baseline ledger must distinguish a fitted model, a failed training attempt, a deferred extension and an unavailable published-method adaptation. The strongest paper is the one its completed evidence supports, even if a simple predictor wins.

In [ ]:
show_status(run, "auxiliary/protocol.json")
display(pd.json_normalize(fold_json_records(run, "tabular_audit.json")))
optional_evidence(run, "seed_sensitivity.csv", "frozen-recipe repetition across training seeds")

## 6. Keep the five-condition classifier in a separate table

The Random Forest and any encoder probe receive condition labels from permitted non-test training sources. This is a supervised five-label task. The normal-only anomaly detector does not receive that supervision, so the two results should not compete in one unlabeled accuracy table.

The exp5 comparison requires the same selected conditions, source splits and input provenance. Report balanced accuracy, macro-F1, class support and confusion matrices. A source with more than one condition membership needs an explicit handling rule; it cannot become several independent people. Tiny disease-family counts remain descriptive.

In [ ]:
condition_results = pd.json_normalize(run.json("five_class_metrics.json"))
display(condition_results)
display(condition_results.groupby("method")[["macro_f1", "balanced_accuracy"]].mean())
print("These means describe four held-out source folds; their variability is not a clinical confidence interval.")
display(plot_condition_confusions(run)); plt.close("all")

## 7. State the generalization boundary with the result

GAVD offers varied real web video but has uncontrolled camera view, age, speed, clothing, aids, diagnostic verification and possible repeated identities. Many windows from one video increase measurement detail without increasing independent participants. Excluding poor poses can shift the observed population, including away from severe or unusual movement.

The study can support a source-level result under its recorded acquisition and eligibility rules. It cannot establish clinical diagnosis, new-site performance or neurological mechanism. A bounded independent laboratory replication would strengthen the measurement claim, but it must be reported with its own participants, sensors and fitted references.

## Preserve the evidence trail

The run directory is the unit of reproducibility. Keep its input hashes, source roles, fitted models, score references, target predictions, source scores, verification record and failed-comparison ledger together. A successful notebook execution establishes that this tutorial can reproduce its displayed analysis. It does not establish a neurological cause or a clinically valid classifier.

The [iteration02 protocol](docs/iteration02/protocol.md), [results](docs/iteration02/results.md), [limitations](docs/iteration02/limitations.md), and [draft02 readiness review](docs/draft02/README.md) must agree with the same run. The earlier draft01 and notebooks 00–11 remain historical evidence. New figures are written as SVG and PDF masters with PNG previews and an input-hash sidecar under this run's `figures/notebooks` directory.

In [ ]:
record_path = run.record('16_baselines_and_source_level_evaluation')
print("Recorded the exact real artifacts read by this notebook:", record_path)
print("This notebook did not replace the experiment producer or alter its source roles.")